In [5]:
# PROGRAM 1 : RNN - Word-level Text Generation

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer

# --- Corpus ---
text = """
the weather today is sunny and warm
the weather tomorrow will be cloudy
it is raining heavily outside today
the sun is shining bright in the sky
cold winds are blowing from the north
the temperature is rising in summer
winter brings snow and cold weather
spring flowers bloom in warm weather
the sky is clear and blue today
storms bring thunder in the evening
warm sunny days are good for activities
the clouds are moving fast in the sky
today the wind is blowing very strongly
the morning is cold and the evening is warm
rain is falling softly in the garden
the sun rises early in the summer morning
clouds cover the sky in the afternoon
the weather is pleasant in the spring
snow falls gently during winter nights
the breeze is cool and refreshing today
""".lower().strip()

# --- Tokenization ---
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
sequences = tokenizer.texts_to_sequences([text])[0]

vocab_size = len(tokenizer.word_index) + 1
word2idx = tokenizer.word_index
idx2word = {v: k for k, v in word2idx.items()}

print(f"Vocabulary size  : {vocab_size}")
print(f"Total tokens     : {len(sequences)}")

# --- Prepare Training Data ---
seq_len = 5
X, y = [], []

for i in range(len(sequences) - seq_len):
    X.append(sequences[i:i+seq_len])
    y.append(sequences[i+seq_len])

X = np.array(X)
y = to_categorical(np.array(y), num_classes=vocab_size)

print(f"Training samples : {len(X)}\n")

# --- Build Model ---
model = Sequential([
    Embedding(vocab_size, 32, input_length=seq_len),
    SimpleRNN(64, activation='tanh'),
    Dense(64, activation='relu'),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

# --- Train ---
history = model.fit(X, y, epochs=300, batch_size=16, verbose=0)

print("\nTraining Progress:")
for ep in [100, 200, 300]:
    print(f"Epoch {ep} — Loss: {history.history['loss'][ep-1]:.4f} | Accuracy: {history.history['accuracy'][ep-1]:.4f}")

print(f"\nFinal Accuracy: {history.history['accuracy'][-1]:.4f}")

# --- Text Generator ---
def generate_text(seed_words, num_words=8, temperature=0.5):
    result = seed_words.lower().split()

    for _ in range(num_words):
        chunk = result[-seq_len:]

        while len(chunk) < seq_len:
            chunk = ['the'] + chunk

        x = np.array([[word2idx.get(w, 1) for w in chunk]])

        pred = model.predict(x, verbose=0)[0]

        # Temperature sampling
        pred = np.log(pred + 1e-10) / temperature
        pred = np.exp(pred) / np.sum(np.exp(pred))

        next_idx = np.random.choice(len(pred), p=pred)
        next_word = idx2word.get(next_idx, 'the')

        result.append(next_word)

    return ' '.join(result)

# TEST CASES

test_cases = [
    ("the weather is", 0.3),
    ("today the weather", 0.5),
    ("the sky is", 0.4),
    ("it is very", 0.6),
    ("the sun rises", 0.3),
    ("rain is falling", 0.5),
    ("cold winds are blowing", 0.3),
    ("the temperature is", 0.4),
    ("snow falls", 0.3),
    ("the morning is", 0.5),
]

print("        RNN WORD-LEVEL TEXT GENERATION RESULTS")

for seed, temp in test_cases:
    out = generate_text(seed, num_words=7, temperature=temp)
    print(f"\nTemp : {temp}")
    print(f"Input : '{seed}'")
    print(f"Output: '{out}'")


Vocabulary size  : 69
Total tokens     : 138
Training samples : 133



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training Progress:
Epoch 100 — Loss: 0.0140 | Accuracy: 1.0000
Epoch 200 — Loss: 0.0027 | Accuracy: 1.0000
Epoch 300 — Loss: 0.0010 | Accuracy: 1.0000

Final Accuracy: 1.0000
        RNN WORD-LEVEL TEXT GENERATION RESULTS

Temp : 0.3
Input : 'the weather is'
Output: 'the weather is early the the spring weather rises strongly'

Temp : 0.5
Input : 'today the weather'
Output: 'today the weather is warm very the spring is rising'

Temp : 0.4
Input : 'the sky is'
Output: 'the sky is early the blue today weather days is'

Temp : 0.6
Input : 'it is very'
Output: 'it is very be strongly today sky cold cold summer'

Temp : 0.3
Input : 'the sun rises'
Output: 'the sun rises early in the summer morning clouds cover'

Temp : 0.5
Input : 'rain is falling'
Output: 'rain is falling in the the is the cover is'

Temp : 0.3
Input : 'cold winds are blowing'
Output: 'cold winds are blowing is the north the the softly is'

Temp : 0.4
Input : 'the temperature is'
Output: 'the temperature is early the summe

In [6]:
# PROGRAM 2: LSTM - Sentiment Analysis

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

sentences = [
    "This movie was absolutely fantastic", "I loved every moment of it",
    "Great film highly recommend", "Amazing acting and storyline",
    "The best movie I have ever seen", "Brilliant and entertaining",
    "This was a terrible movie", "I hated it completely boring",
    "Worst film ever waste of time", "Awful acting bad storyline",
    "I did not enjoy this at all", "Disappointing and dull"
]
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

tokenizer = Tokenizer(num_words=500, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=10, padding='post', truncating='post')

X = np.array(padded)
y = np.array(labels)

model = Sequential([
    Embedding(500, 16, input_length=10),
    LSTM(32),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(X, y, epochs=50, verbose=0)
print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")

def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=10, padding='post')
    score = model.predict(pad, verbose=0)[0][0]
    label = "Positive" if score >= 0.5 else "Negative"
    return label, score

test_sentences = [
    "This movie was absolutely fantastic",
    "Worst film ever waste of time"
]
print("\n--- Predictions ---")
for s in test_sentences:
    label, score = predict_sentiment(s)
    print(f"Input   : '{s}'")
    print(f"Output  : {label} ({score:.4f})\n")

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Final Training Accuracy: 1.0000

--- Predictions ---
Input   : 'This movie was absolutely fantastic'
Output  : Positive (0.5995)

Input   : 'Worst film ever waste of time'
Output  : Negative (0.1653)



In [7]:
# PROGRAM 3: BERT - Text Classification using Hugging Face

!pip install transformers -q

from transformers import pipeline

print("Loading BERT model (bert-base-uncased fine-tuned)...")
classifier = pipeline(
    "text-classification",
    model="textattack/bert-base-uncased-imdb",
    tokenizer="textattack/bert-base-uncased-imdb"
)

test_sentences = [
    "I love this product, it works great!",
    "This is the worst experience I have ever had.",
    "The food was okay, nothing special.",
    "Absolutely brilliant performance by the entire cast!"
]

print("\n--- BERT Predictions ---")
print(f"{'Input':<50} {'Label':<12} {'Score':<8}")
print("-" * 75)
for sentence in test_sentences:
    result = classifier(sentence)[0]
    label = "POSITIVE" if result['label'] == 'LABEL_1' else "NEGATIVE"
    print(f"{sentence[:48]:<50} {label:<12} {result['score']:.4f}")

Loading BERT model (bert-base-uncased fine-tuned)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- BERT Predictions ---
Input                                              Label        Score   
---------------------------------------------------------------------------
I love this product, it works great!               POSITIVE     0.9801
This is the worst experience I have ever had.      NEGATIVE     0.9989
The food was okay, nothing special.                NEGATIVE     0.7042
Absolutely brilliant performance by the entire c   POSITIVE     0.9987


In [8]:
# PROGRAM 4: RoBERTa - Sentiment Analysis using Hugging Face

!pip install transformers -q

from transformers import pipeline

print("Loading RoBERTa model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

test_sentences = [
    "I love this product, it works great!",
    "The service was terrible and I am very disappointed.",
    "The weather is nice today.",
    "I cannot believe how bad this is!"
]

label_map = {"positive": "POSITIVE", "neutral": "NEUTRAL", "negative": "NEGATIVE"}

print("\n--- RoBERTa Predictions ---")
print(f"{'Input':<50} {'Label':<12} {'Score':<8}")
print("-" * 75)
for sentence in test_sentences:
    result = sentiment_pipeline(sentence)[0]
    label = label_map.get(result['label'].lower(), result['label'].upper())
    print(f"{sentence[:48]:<50} {label:<12} {result['score']:.4f}")

Loading RoBERTa model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- RoBERTa Predictions ---
Input                                              Label        Score   
---------------------------------------------------------------------------
I love this product, it works great!               POSITIVE     0.9882
The service was terrible and I am very disappoin   NEGATIVE     0.9429
The weather is nice today.                         POSITIVE     0.9819
I cannot believe how bad this is!                  NEGATIVE     0.9414
